# Data Preparation

Prepare the encounter data for analysis. This section applies basic cleaning and audits the resulting CSV.

## Setup and Raw Data Loading

Load the verified source encounters from the local raw CSV. The loader validates the input file, schema, and original labels.

In [ ]:
import sys
from pathlib import Path

from IPython.display import Markdown, display

project_root = next(
    directory
    for directory in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (directory / "src" / "paths.py").is_file() and (directory / "requirements.txt").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preparation.data_cleaning import clean_encounters, save_cleaning_result
from src.preparation.data_loading import load_raw_encounters

encounters = load_raw_encounters()

## Basic Data Cleaning

Apply the configured marker replacements. Each configured marker becomes an empty CSV field. The laboratory category `None` retains its literal representation.

In [ ]:
cleaned = clean_encounters(encounters)
changed_cells = sum(row["changed_cells"] for row in cleaned.audit)
print(f"Encounters: {len(encounters.rows):,} before, {len(cleaned.rows):,} after")
print(f"Columns: {len(encounters.columns):,} before, {len(cleaned.columns):,} after")
print(f"Changed cells: {changed_cells:,}")

audit_table = [
    "| Column | Source marker | Markers before | Empty fields before | Empty fields after | Literal retained |",
    "| --- | --- | ---: | ---: | ---: | --- |",
]
for row in cleaned.audit:
    if row["source_marker"] or row["preserved_literal"]:
        marker = f"`{row['source_marker']}`" if row["source_marker"] else ""
        literal = f"`{row['preserved_literal']}` ({row['literal_after']:,})" if row["preserved_literal"] else ""
        audit_table.append(
            f"| `{row['column']}` | {marker} | {row['marker_before']:,} | "
            f"{row['empty_before']:,} | {row['empty_after']:,} | {literal} |"
        )
display(Markdown("\n".join(audit_table)))

In [ ]:
cleaned_path, audit_path = save_cleaning_result(cleaned)
print(cleaned_path.relative_to(project_root))
print(audit_path.relative_to(project_root))